<a href="https://colab.research.google.com/github/murilo-guimaraes/redes-computadores/blob/main/Ethernet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
#**CAMADA DE REDE: PROTOCOLOS E ALGORITMOS DE ROTEAMENTO**

####**Relatório Técnico Acadêmico | Redes 2026 | Colaboratory**
**Data: 23/04/2026**
<div>Roteamento Dinâmico | Tabela de Roteamento | Vetor de Distância | Estado de Link</div>

*Baseado na Unidade 3 Seção 2 do livro **Redes de Computadores** (Sergio Eduardo Nunes)*

##**Introdução Teórica**

O roteamento é a função primordial da camada de rede, responsável por determinar o caminho que os pacotes devem seguir da origem ao destino. Segundo Nunes (2017), os **Roteadores** utilizam tabelas de roteamento para tomar decisões de encaminhamento. Estas tabelas podem ser preenchidas de forma estática ou dinâmica, através de protocolos de roteamento. Protocolos como o **RIP** (baseado em Vetor de Distância) e o **OSPF** (baseado em Estado de Link) permitem que os roteadores aprendam a topologia da rede e reajam a falhas de forma autônoma, garantindo a resiliência da comunicação inter-redes.

---
##**Objetivos**

 - **Analisar** a tabela de roteamento do sistema local para entender o encaminhamento de pacotes.
 - **Simular** o funcionamento de um algoritmo de vetor de distância para escolha de melhor caminho.
 - **Demonstrar** a métrica de "salto" (hop) e como ela influencia a topologia da rede.
 - **Validar** a importância da gerência de rotas em redes complexas.

---
### **1. Inspeção da Tabela de Roteamento do Sistema**

Nesta etapa, utilizaremos comandos de terminal para visualizar as rotas ativas no kernel. Para que o Sistema Operacional saiba para onde enviar um pacote, ele consulta uma tabela interna que define o próximo salto (Next Hop) com base no destino, conforme a lógica explicada por Nunes.

In [ ]:
# Comando para listar a tabela de roteamento IP do kernel
!ip route show

default via 172.28.0.1 dev eth0 
172.28.0.0/16 dev eth0 proto kernel scope link src 172.28.0.12 


#### **O que observar:**
<a>O output exibe a "default via", que identifica o Gateway Padrão mencionado na Seção 3.2. A tabela mostra as redes conhecidas pelo sistema e para qual interface ou roteador o tráfego deve ser direcionado. Como Nunes descreve, cada entrada representa uma decisão lógica que garante que o pacote percorra o caminho correto na sub-rede.</a>

---
### **2. Simulação de Algoritmo de Melhor Caminho (Métrica de Salto)**

Os protocolos de roteamento dinâmico calculam o custo de um caminho para evitar congestionamentos. Vamos simular um algoritmo de Vetor de Distância (RIP), onde o roteador escolhe a rota com o menor número de "saltos" (hops) até o destino final.

In [ ]:
# Simulação de escolha de rota baseada em métrica de Saltos
rotas_disponiveis = {
    "Caminho_A": {"saltos": 5, "latencia": "50ms"},
    "Caminho_B": {"saltos": 2, "latencia": "120ms"},
    "Caminho_C": {"saltos": 4, "latencia": "30ms"}
}

print("[SISTEMA] Analisando melhores rotas para o destino...")

# O algoritmo de Vetor de Distância foca puramente no número de saltos
melhor_rota = min(rotas_disponiveis, key=lambda x: rotas_disponiveis[x]['saltos'])

print(f"\n[DECISÃO] A melhor rota escolhida foi: {melhor_rota}")
print(f"Detalhes: {rotas_disponiveis[melhor_rota]['saltos']} saltos detectados.")

[SISTEMA] Analisando melhores rotas para o destino...

[DECISÃO] A melhor rota escolhida foi: Caminho_B
Detalhes: 2 saltos detectados.


#### **O que observar:**
<a>Neste experimento, o algoritmo escolhe o Caminho B por possuir o menor número de saltos. Isso ilustra fielmente a lógica do protocolo RIP descrita por Nunes, onde a distância é a métrica principal.
>**Nota Técnica:** Como não temos múltiplos roteadores físicos no ambiente Colab para alterar rotas em tempo real, utilizamos esta simulação em Python para demonstrar a tomada de decisão do algoritmo de roteamento.</a>

---
### **3. Rastreamento de Saltos entre Redes (Traceroute)**

Na Seção 3.2, Nunes explica que um pacote raramente chega ao destino em um único passo, passando por diversos roteadores intermediários (saltos). Nesta etapa, utilizaremos a ferramenta de diagnóstico `traceroute` para rastrear a rota de um pacote, identificando os pontos de roteamento e como o sistema lida com a latência e a segurança no caminho.

In [ ]:
# Instalando a ferramenta de traceroute no ambiente Linux do Colab
!apt-get install traceroute -y > /dev/null

# Executando o rastreio para um servidor externo
# O parâmetro -I tenta usar pacotes ICMP
!traceroute -I 8.8.8.8

traceroute to 8.8.8.8 (8.8.8.8), 30 hops max, 60 byte packets
 1  172.28.0.1 (172.28.0.1)  0.044 ms  0.009 ms  0.007 ms
 2  * * *
 3  * * *
 4  * * *
 5  * * *
 6  * * *
 7  * * *
 8  * * *
 9  * * *
10  * * *
11  * * *
12  * * *
13  * * *
14  dns.google (8.8.8.8)  0.441 ms  0.410 ms  0.405 ms


#### **O que observar:**
<a>O output lista os endereços IP dos roteadores que aceitaram o pacote.
> **Nota Técnica:** É comum que alguns saltos (hops) apareçam apenas com asteriscos (`* * *`). Isso ocorre por motivos de segurança do ambiente e dos roteadores intermediários, que são configurados para não responder a pacotes de diagnóstico (ICMP/UDP) para evitar ataques de mapeamento de rede. Apesar da ausência de resposta visual desses nós, a teoria de Nunes se valida pelo fato de o pacote continuar sendo encaminhado até o destino final ou até o limite de saltos permitido.</a>

---
##**Conclusão**

A prática baseada na Unidade 3 Seção 3.2 confirmou que o roteamento é a inteligência da camada de rede. Através da inspeção de tabelas e do uso do traceroute, ficou claro que a escolha do caminho ideal e a resposta dos dispositivos dependem de políticas de segurança e métricas bem definidas. O entendimento de que nem todo roteador "se revela" no caminho (conforme visto nos asteriscos do teste) é fundamental para o profissional de ADS compreender a segurança e a resiliência da infraestrutura global detalhada por Sergio Eduardo Nunes.

---
##**Referências Bibliográficas**

NUNES, Sergio Eduardo. **Redes de computadores**. Londrina: Editora e Distribuidora Educacional S.A., 2017.

KUROSE, James F.; ROSS, Keith W. **Redes de Computadores e a Internet**: uma abordagem top-down. 6. ed. São Paulo: Pearson, 2013.

TANENBAUM, Andrew S.; WETHERALL, David. **Redes de Computadores**. 5. ed. São Paulo: Pearson, 2011.

---
<p align="center"><b>© 2026 Murilo Guimarães. Acadêmico de ADS.</b></p>

---